In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# قياس الكيوبتات

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
للحصول على معلومات حول حالة الكيوبت، يمكنك _قياسه_ على [بت كلاسيكي](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit). في Qiskit، تُجرى القياسات في الأساس الحسابي، أي أساس Pauli-$Z$ أحادي الكيوبت. لذلك، يُعطي القياس القيمة 0 أو 1، تبعاً للتداخل مع الحالات الذاتية لـ Pauli-$Z$ وهي $|0\rangle$ و$|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## قياسات منتصف الدائرة
قياسات منتصف الدائرة هي مكوّن أساسي في الدوائر الديناميكية. قبل الإصدار v0.43.0 من `qiskit-ibm-runtime`، كانت `measure` هي تعليمة القياس الوحيدة في Qiskit. غير أن قياسات منتصف الدائرة تختلف في متطلبات الضبط عن القياسات _الطرفية_ (القياسات التي تحدث في نهاية الدائرة). فمثلاً، تحتاج إلى مراعاة مدة التعليمة عند ضبط قياس منتصف الدائرة، لأن التعليمات الأطول تُنتج دوائر أكثر ضوضاءً. في المقابل، لا تحتاج إلى مراعاة مدة التعليمة في القياسات الطرفية لأنه لا توجد تعليمات بعدها.

في الإصدار v0.43.0 من `qiskit-ibm-runtime`، تم تقديم تعليمة `MidCircuitMeasure`. كما يوحي الاسم، هي تعليمة قياس جديدة مُحسَّنة لقياسات منتصف الدائرة على أجهزة IBM&reg; QPU.

> **Note:** تعليمة `MidCircuitMeasure` تُعيَّن إلى تعليمة `measure_2` المُبلَّغ عنها في `supported_instructions` الخاصة بالجهاز الخلفي. إلا أن `measure_2` غير مدعومة على جميع الأجهزة الخلفية. استخدم `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` للعثور على الأجهزة التي تدعمها. قد تُضاف قياسات جديدة في المستقبل، لكن هذا غير مضمون.

## تطبيق قياس على دائرة
هناك عدة طرق لتطبيق القياسات على دائرة:

### طريقة `QuantumCircuit.measure`
استخدم طريقة [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) لقياس [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).

أمثلة:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### صنف `Measure`
يقوم صنف Qiskit [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) بقياس الكيوبتات المحددة.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### طريقة `QuantumCircuit.measure_all`
لقياس جميع الكيوبتات في البتات الكلاسيكية المقابلة لها، استخدم طريقة [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all). بشكل افتراضي، تضيف هذه الطريقة بتات كلاسيكية جديدة في `ClassicalRegister` لتخزين هذه القياسات.